In [1]:
# =============================================================================
# MedSSL-3D — قدم ۱: بررسی پراکندگی شدت MRI بین بیماران
#
# این را در یک سلول *جدید* اجرا کنید. هیچ فایل نوت‌بوکی را باز نکنید و
# هیچ چیزی را تغییر ندهید — این اسکریپت کاملاً مستقل و فقط-خواندنی است.
# چیزی روی دیسک نمی‌نویسد و به هیچ چک‌پوینتی دست نمی‌زند.
#
# سؤالی که جواب می‌دهد:
#   «شدت خام MRI بین بیماران چقدر فرق می‌کند، و بنابراین نبودِ
#    NormalizeIntensity در پایپ‌لاین segmentation چقدر آسیب‌زاست؟»
# =============================================================================

import os
import glob
import numpy as np
import nibabel as nib

DATA_ROOT   = 'D:/MedSSL/MICCAI_BraTS2020_TrainingData'
N_PATIENTS  = 15                                  # ۱۵ بیمار: کمتر از یک دقیقه
MODALITIES  = ('flair', 't1', 't1ce', 't2')       # همان ترتیب MODALITY_ORDER

# ---- پیدا کردن پوشه‌های بیمار ------------------------------------------------
patient_dirs = sorted(
    d for d in glob.glob(os.path.join(DATA_ROOT, 'BraTS20_Training_*'))
    if os.path.isdir(d)
)
if not patient_dirs:
    raise SystemExit(f'🔴 هیچ پوشه‌ی بیماری در {DATA_ROOT} پیدا نشد.')

# نمونه‌گیری یکنواخت در کل دیتاست، نه فقط ۱۵ تای اول — بیماران متوالی
# ممکن است از یک مرکز/اسکنر باشند و پراکندگی واقعی را پنهان کنند.
idx = np.linspace(0, len(patient_dirs) - 1, min(N_PATIENTS, len(patient_dirs))).astype(int)
selected = [patient_dirs[i] for i in idx]

print(f'📂 {len(patient_dirs)} بیمار پیدا شد | نمونه‌گیری {len(selected)} بیمار به‌صورت یکنواخت\n')


def load_volume(pdir, modality):
    """بارگذاری یک مدالیته؛ هم .nii و هم .nii.gz را پوشش می‌دهد."""
    pid = os.path.basename(pdir)
    for ext in ('.nii', '.nii.gz'):
        path = os.path.join(pdir, f'{pid}_{modality}{ext}')
        if os.path.isfile(path):
            return np.asarray(nib.load(path).dataobj, dtype=np.float32)
    return None


# ---- جمع‌آوری آمار ------------------------------------------------------------
# فقط voxelهای غیرصفر: BraTS دیتای skull-strip شده است، پس پس‌زمینه دقیقاً صفر
# است و اگر آن را وارد میانگین کنیم، تفاوت واقعی بافت پنهان می‌شود.
# این دقیقاً همان قراردادی است که NormalizeIntensity(nonzero=True) استفاده می‌کند.
stats = {m: [] for m in MODALITIES}
shapes = set()

for pdir in selected:
    pid = os.path.basename(pdir)
    row = [f'{pid:<24}']
    for m in MODALITIES:
        vol = load_volume(pdir, m)
        if vol is None:
            row.append(f'{m}: ---')
            continue
        shapes.add(vol.shape)
        nz = vol[vol > 0]
        if nz.size == 0:
            continue
        stats[m].append({'pid': pid, 'mean': float(nz.mean()),
                         'std': float(nz.std()), 'max': float(nz.max())})
        row.append(f'{m}: mean={nz.mean():8.1f} std={nz.std():7.1f} max={nz.max():8.1f}')
    print('  ' + ' | '.join(row))

print(f'\n📐 shapeهای مشاهده‌شده (خام، قبل از Resize): {shapes}')

# ---- عدد کلیدی: نسبت پراکندگی بین بیماران -----------------------------------
print('\n' + '=' * 78)
print('📊 پراکندگی میانگین شدت بین بیماران (فقط voxelهای غیرصفر)')
print('=' * 78)
print(f"{'مدالیته':<10}{'کمینه‌ی mean':>14}{'بیشینه‌ی mean':>15}{'نسبت':>10}   حکم")

worst_ratio = 0.0
for m in MODALITIES:
    if not stats[m]:
        continue
    means = np.array([s['mean'] for s in stats[m]])
    lo, hi = means.min(), means.max()
    ratio = hi / lo if lo > 0 else float('inf')
    worst_ratio = max(worst_ratio, ratio)
    if ratio >= 2.0:
        verdict = '🔴 پراکندگی زیاد — normalization لازم است'
    elif ratio >= 1.3:
        verdict = '🟡 پراکندگی متوسط'
    else:
        verdict = '🟢 پراکندگی کم'
    print(f'{m:<10}{lo:>14.1f}{hi:>15.1f}{ratio:>9.2f}x   {verdict}')

# ---- اثبات اینکه normalization مشکل را حل می‌کند ------------------------------
# اعمال همان قاعده‌ای که NormalizeIntensity(nonzero=True, channel_wise=True)
# اجرا می‌کند: z-score روی voxelهای غیرصفر، جدا برای هر کانال.
print('\n' + '=' * 78)
print('🔬 همان بیماران، بعد از z-score روی voxelهای غیرصفر (کاری که Mode A می‌کند)')
print('=' * 78)
for m in MODALITIES:
    if not stats[m]:
        continue
    # بعد از z-score، به‌تعریف میانگینِ voxelهای غیرصفر صفر و انحراف معیار یک می‌شود
    # برای هر بیمار — پس نسبت بین‌بیماری دقیقاً ۱.۰۰ می‌شود.
    print(f'{m:<10} همه‌ی بیماران -> mean=0.00  std=1.00  |  نسبت بین‌بیماری = 1.00x  🟢')

# ---- جمع‌بندی ----------------------------------------------------------------
print('\n' + '=' * 78)
print('🏁 نتیجه')
print('=' * 78)
print(f'   بدترین نسبت پراکندگی بین بیماران: {worst_ratio:.2f}x')
if worst_ratio >= 2.0:
    print('   🔴 تأیید شد: شبکه در پایپ‌لاین segmentation با مقیاس‌های ورودی')
    print(f'      تا {worst_ratio:.1f} برابر متفاوت مواجه می‌شود. با batch_size=2،')
    print('      BatchNorm نمی‌تواند این را جبران کند (روی batch نرمال می‌کند، نه هر نمونه).')
elif worst_ratio >= 1.3:
    print('   🟡 پراکندگی متوسط — بهبود انتظار می‌رود ولی نه جهش بزرگ.')
else:
    print('   🟢 پراکندگی کم — سود مستقیم normalization احتمالاً کم است.')

print('\n   ⚠️ مستقل از عدد بالا: Mode A روی ورودی z-score شده آموزش دیده و')
print('      Mode B/C همان encoder را روی ورودی خام اعمال می‌کند. این ناسازگاری')
print('      یک واقعیت کد است و صرف‌نظر از نتیجه‌ی این تست باید رفع شود.')
print('\n   ✅ این اسکریپت فقط خواند. هیچ فایلی نوشته یا تغییر داده نشد.')

📂 369 بیمار پیدا شد | نمونه‌گیری 15 بیمار به‌صورت یکنواخت

  BraTS20_Training_001     | flair: mean=   173.0 std=   64.9 max=   625.0 | t1: mean=   354.3 std=   84.2 max=   678.0 | t1ce: mean=   417.3 std=  109.2 max=  1845.0 | t2: mean=   114.7 std=   47.7 max=   376.0
  BraTS20_Training_027     | flair: mean=   266.4 std=   98.2 max=   973.0 | t1: mean=   328.4 std=   84.9 max=   581.0 | t1ce: mean=   363.5 std=   97.4 max=  1393.0 | t2: mean=   172.9 std=   69.0 max=   607.0
  BraTS20_Training_053     | flair: mean=   190.5 std=   70.1 max=   625.0 | t1: mean=   320.9 std=   73.5 max=   606.0 | t1ce: mean=   369.0 std=  114.6 max=  1883.0 | t2: mean=   440.0 std=  177.8 max=  1663.0
  BraTS20_Training_079     | flair: mean=   129.7 std=   37.0 max=   517.0 | t1: mean=   307.8 std=   79.2 max=   546.0 | t1ce: mean=   364.1 std=   93.5 max=  1418.0 | t2: mean=   114.3 std=   50.7 max=   417.0
  BraTS20_Training_106     | flair: mean=   137.3 std=   59.1 max=   720.0 | t1: mean=   285.